# Modelling panel — EDA and preparation

Panel-level cleaning and prep on the supervised spatial–temporal units from notebook `10`.

| Item | Detail |
|------|--------|
| Input | `data/03_processed/modelling_panel.parquet` + `modelling_panel_manifest.json` |
| Grain | Tier 1: `(h3_cell, year, month, vector_group)`; Tier 2: `(country_iso2, year, month, vector_group)` |
| Target | `virus_detected` (binary); `virus_detection_count` retained |
| Training filter (upstream) | VectorNet coverage required — virus-only units already excluded |
| Spec | [`specs/04-reference-data-and-panel-pipeline.md`](specs/04-reference-data-and-panel-pipeline.md) |

**Prerequisite:** run [`notebooks/01_sources/10_merge_modelling_panel.ipynb`](notebooks/01_sources/10_merge_modelling_panel.ipynb) so the processed panel exists.

This notebook covers load → profile → missingness / outliers → geo/time filters → modelling handoff. Distribution plots, target deep-dive, feature importance, and model training come in later cells.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from vector_guard.data import default_processed_dir, find_repo_root
from vector_guard.resolve import H3_RESOLUTION

REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

# Modelling window: denser VectorNet era; all current virus-positive units fall in 2012–2023.
YEAR_MIN = 2010
YEAR_MAX = None  # inclusive upper bound; None = no cap

# Winsorize heavy-tailed vector effort proxies at this quantile (positives sit well below p99).
COUNT_WINSOR_Q = 0.99
COUNT_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
]

IDENTITY_COLS = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
]
TIER1_KEY = ["h3_cell", "year", "month", "vector_group"]
TIER2_KEY = ["country_iso2", "year", "month", "vector_group"]

TARGET_COL = "virus_detected"
FEATURE_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "virus_source_n_sequence",
    "virus_source_n_outbreak",
    "share_virus_exact_coords",
    "share_virus_subnational_centroid",
]

{
    "REPO_ROOT": str(REPO_ROOT),
    "PANEL_PATH": str(PANEL_PATH.relative_to(REPO_ROOT)),
    "H3_RESOLUTION": H3_RESOLUTION,
    "YEAR_MIN": YEAR_MIN,
}

{'REPO_ROOT': '/workspaces/vector-guard',
 'PANEL_PATH': 'data/03_processed/modelling_panel.parquet',
 'H3_RESOLUTION': 4,
 'YEAR_MIN': 2010}

## Load panel and merge manifest

In [2]:
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PANEL_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )

panel = pd.read_parquet(PANEL_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}

{
    "rows": len(panel),
    "cols": len(panel.columns),
    "manifest_panel_rows": manifest.get("panel_rows"),
    "manifest_date_range": manifest.get("date_range"),
    "virus_positive_units": manifest.get("virus_positive_units"),
    "virus_positive_rate": manifest.get("virus_positive_rate"),
    "rows_by_spatial_tier": manifest.get("rows_by_spatial_tier"),
    "training_filter": manifest.get("training_filter"),
}

{'rows': 12995,
 'cols': 16,
 'manifest_panel_rows': 12995,
 'manifest_date_range': {'max': '2025-04', 'min': '1957-09'},
 'virus_positive_units': 35,
 'virus_positive_rate': 0.0026933435936898806,
 'rows_by_spatial_tier': {'h3': 12995},
 'training_filter': 'Supervised panel includes only units with VectorNet coverage; virus-only units are excluded. Virus zeros mean no reported detection in the merged feeds for that unit-window.'}

## Panel profile

Shape, dtypes, grain uniqueness, and basic value ranges.

In [3]:
display(panel.dtypes.to_frame("dtype"))
display(panel.head(5))

tier1 = panel.loc[panel["spatial_tier"] == "h3"]
tier2 = panel.loc[panel["spatial_tier"] == "country"]

profile = {
    "n_rows": len(panel),
    "n_tier1_h3": len(tier1),
    "n_tier2_country": len(tier2),
    "dup_tier1_keys": int(tier1.duplicated(TIER1_KEY).sum()) if len(tier1) else 0,
    "dup_tier2_keys": int(tier2.duplicated(TIER2_KEY).sum()) if len(tier2) else 0,
    "year_min": int(panel["year"].min()),
    "year_max": int(panel["year"].max()),
    "n_h3_cells": int(panel["h3_cell"].nunique(dropna=True)),
    "n_countries": int(panel["country_iso2"].nunique(dropna=True)),
    "vector_groups": sorted(panel["vector_group"].dropna().unique().tolist()),
    "vector_present_all_one": bool((panel["vector_present"] == 1).all()),
    "virus_positive": int(panel[TARGET_COL].sum()),
    "virus_positive_rate": float(panel[TARGET_COL].mean()),
}
profile

,dtype
spatial_tier,str
h3_cell,str
country_iso2,str
year,int64
month,int64
vector_group,str
vector_present,int64
vector_detection_count,int64
individual_count_sum,int64
individual_count_mean,float64


,spatial_tier,h3_cell,country_iso2,year,month,vector_group,vector_present,vector_detection_count,individual_count_sum,individual_count_mean,virus_detected,virus_detection_count,virus_source_n_sequence,virus_source_n_outbreak,share_virus_exact_coords,share_virus_subnational_centroid
0,h3,8419519ffffffff,GB,1974,3,biting_midge,1,14,11,0.785714,0,0,0,0,NaN,NaN
1,h3,84195b7ffffffff,GB,1974,3,biting_midge,1,14,38,2.714286,0,0,0,0,NaN,NaN
2,h3,8419519ffffffff,GB,1974,4,biting_midge,1,14,15,1.071429,0,0,0,0,NaN,NaN
3,h3,84195b7ffffffff,GB,1974,4,biting_midge,1,21,9,0.428571,0,0,0,0,NaN,NaN
4,h3,8419519ffffffff,GB,1974,5,biting_midge,1,14,1,0.071429,0,0,0,0,NaN,NaN


{'n_rows': 12995,
 'n_tier1_h3': 12995,
 'n_tier2_country': 0,
 'dup_tier1_keys': 0,
 'dup_tier2_keys': 0,
 'year_min': 1957,
 'year_max': 2025,
 'n_h3_cells': 2140,
 'n_countries': 57,
 'vector_groups': ['biting_midge', 'mosquito', 'sandfly', 'tick'],
 'vector_present_all_one': True,
 'virus_positive': 35,
 'virus_positive_rate': 0.0026933435936898806}

In [4]:
display(
    panel.groupby("vector_group", dropna=False)
    .agg(
        n_units=("vector_present", "size"),
        virus_positive=(TARGET_COL, "sum"),
        mean_vector_detections=("vector_detection_count", "mean"),
    )
    .assign(positive_rate=lambda d: d["virus_positive"] / d["n_units"])
)

display(
    panel.groupby("year", dropna=False)
    .agg(n_units=("vector_present", "size"), virus_positive=(TARGET_COL, "sum"))
    .tail(20)
)

,n_units,virus_positive,mean_vector_detections,positive_rate
vector_group,,,,
biting_midge,5038,7,9.651052,0.001389
mosquito,6674,28,7.206173,0.004195
sandfly,425,0,42.324706,0.000000
tick,858,0,5.199301,0.000000


,n_units,virus_positive
year,,
2006,316,0
2007,230,0
2008,163,0
2009,181,0
2010,349,0
2011,386,0
2012,343,4
2013,512,0
2014,472,0


## Missing values

Identity, vector features, and labels should be complete on the supervised panel.
`share_virus_*` QA columns are only defined on virus-positive units (NaN elsewhere is expected).

In [5]:
missing = (
    panel.isna()
    .sum()
    .rename("n_missing")
    .to_frame()
    .assign(pct=lambda d: 100.0 * d["n_missing"] / len(panel))
    .sort_values("n_missing", ascending=False)
)
display(missing)

share_cols = ["share_virus_exact_coords", "share_virus_subnational_centroid"]
share_qa = {
    "share_null_when_virus_0": float(
        panel.loc[panel[TARGET_COL] == 0, share_cols[0]].isna().mean()
    ),
    "share_null_when_virus_1": float(
        panel.loc[panel[TARGET_COL] == 1, share_cols[0]].isna().mean()
    ),
}
share_qa

,n_missing,pct
share_virus_subnational_centroid,12960,99.730666
share_virus_exact_coords,12960,99.730666
spatial_tier,0,0.000000
h3_cell,0,0.000000
month,0,0.000000
vector_group,0,0.000000
country_iso2,0,0.000000
year,0,0.000000
vector_detection_count,0,0.000000
vector_present,0,0.000000


{'share_null_when_virus_0': 1.0, 'share_null_when_virus_1': 0.0}

## Outliers (vector effort proxies)

Detection and individual counts are right-skewed. Summarize tails and prepare winsor caps for modelling features (do not drop units — rare virus positives must stay).

In [6]:
outlier_summary = []
winsor_caps: dict[str, float] = {}
for col in COUNT_COLS:
    s = panel[col]
    cap = float(s.quantile(COUNT_WINSOR_Q))
    winsor_caps[col] = cap
    outlier_summary.append(
        {
            "column": col,
            "min": float(s.min()),
            "p50": float(s.median()),
            "p95": float(s.quantile(0.95)),
            f"p{int(COUNT_WINSOR_Q * 100)}": cap,
            "max": float(s.max()),
            "n_above_cap": int((s > cap).sum()),
            "n_pos_above_cap": int(
                ((panel[TARGET_COL] == 1) & (s > cap)).sum()
            ),
        }
    )

display(pd.DataFrame(outlier_summary))
winsor_caps

,column,min,p50,p95,p99,max,n_above_cap,n_pos_above_cap
0,vector_detection_count,1.0,3.0,26.0,80.00,7438.0,128,0
1,individual_count_sum,0.0,12.0,1572.0,9213.36,642933.0,130,0
2,individual_count_mean,0.0,3.0,228.3,1142.31,321466.5,130,0


{'vector_detection_count': 80.0,
 'individual_count_sum': 9213.35999999992,
 'individual_count_mean': 1142.3099999999931}

## Geo / time filters and modelling prep

Applied rules:

1. **Identity completeness** — drop rows missing required key fields for their `spatial_tier`.
2. **Valid calendar month** — `month` in 1–12.
3. **VectorNet coverage** — keep `vector_present == 1` (already true for the merge output).
4. **Modelling year window** — `year >= YEAR_MIN` (and `<= YEAR_MAX` if set).
5. **Feature prep** — winsorize count columns at p99; fill `share_virus_*` NaNs with `0.0` for model matrices (QA meaning unchanged: zero share when no virus rows).

Row drops are reported; virus-positive retention is checked.

In [7]:
def apply_panel_prep(
    df: pd.DataFrame,
    *,
    year_min: int = YEAR_MIN,
    year_max: int | None = YEAR_MAX,
    winsor_q: float = COUNT_WINSOR_Q,
) -> tuple[pd.DataFrame, dict]:
    """Filter and prepare the modelling panel; return (prepared_df, filter_report)."""
    work = df.copy()
    n0 = len(work)
    n_pos0 = int(work[TARGET_COL].sum())
    report: dict = {"n_input": n0, "n_pos_input": n_pos0}

    # 1. Identity by tier
    h3_ok = (work["spatial_tier"] == "h3") & work["h3_cell"].notna()
    country_ok = (work["spatial_tier"] == "country") & work["country_iso2"].notna()
    common_ok = (
        work["spatial_tier"].isin(["h3", "country"])
        & work["year"].notna()
        & work["month"].notna()
        & work["vector_group"].fillna("").ne("")
        & work["country_iso2"].notna()  # metadata preferred on Tier 1 too
    )
    mask_id = common_ok & (h3_ok | country_ok)
    report["dropped_identity"] = int((~mask_id).sum())
    work = work.loc[mask_id].copy()

    # 2. Valid month
    mask_month = work["month"].between(1, 12)
    report["dropped_bad_month"] = int((~mask_month).sum())
    work = work.loc[mask_month].copy()

    # 3. VectorNet coverage
    mask_vec = work["vector_present"] == 1
    report["dropped_no_vector"] = int((~mask_vec).sum())
    work = work.loc[mask_vec].copy()

    # 4. Year window
    mask_year = work["year"] >= year_min
    if year_max is not None:
        mask_year &= work["year"] <= year_max
    report["dropped_year_window"] = int((~mask_year).sum())
    report["year_min"] = year_min
    report["year_max"] = year_max
    work = work.loc[mask_year].copy()

    # 5. Feature prep (winsorize + share fill)
    caps = {c: float(work[c].quantile(winsor_q)) for c in COUNT_COLS}
    report["winsor_caps"] = caps
    report["winsor_q"] = winsor_q
    for col, cap in caps.items():
        work[f"{col}_raw"] = work[col]
        work[col] = work[col].clip(upper=cap)

    for col in share_cols:
        work[col] = work[col].fillna(0.0)

    work["year"] = work["year"].astype(int)
    work["month"] = work["month"].astype(int)

    report["n_output"] = len(work)
    report["n_pos_output"] = int(work[TARGET_COL].sum())
    report["n_dropped_total"] = n0 - len(work)
    report["pos_retained"] = report["n_pos_output"] == n_pos0 or report[
        "n_pos_output"
    ] == int(df.loc[df["year"] >= year_min, TARGET_COL].sum())
    return work.reset_index(drop=True), report


panel_prep, filter_report = apply_panel_prep(panel)
filter_report

{'n_input': 12995,
 'n_pos_input': 35,
 'dropped_identity': 0,
 'dropped_bad_month': 0,
 'dropped_no_vector': 0,
 'dropped_year_window': 3969,
 'year_min': 2010,
 'year_max': None,
 'winsor_caps': {'vector_detection_count': 73.5,
  'individual_count_sum': 10683.75,
  'individual_count_mean': 1200.0},
 'winsor_q': 0.99,
 'n_output': 9026,
 'n_pos_output': 35,
 'n_dropped_total': 3969,
 'pos_retained': True}

In [8]:
prep_missing = (
    panel_prep[IDENTITY_COLS + FEATURE_COLS + [TARGET_COL, "virus_detection_count"]]
    .isna()
    .sum()
)
assert int(prep_missing.sum()) == 0, prep_missing[prep_missing > 0].to_dict()

{
    "prep_rows": len(panel_prep),
    "prep_pos": int(panel_prep[TARGET_COL].sum()),
    "prep_pos_rate": float(panel_prep[TARGET_COL].mean()),
    "prep_year_range": [int(panel_prep["year"].min()), int(panel_prep["year"].max())],
    "prep_missing_feature_cells": int(prep_missing.sum()),
    "spatial_tier_counts": panel_prep["spatial_tier"].value_counts().to_dict(),
}

{'prep_rows': 9026,
 'prep_pos': 35,
 'prep_pos_rate': 0.0038776866829160206,
 'prep_year_range': [2010, 2025],
 'prep_missing_feature_cells': 0,
 'spatial_tier_counts': {'h3': 9026}}

## Modelling handoff

`panel_prep` is the cleaned analysis table for later EDA / training cells.

| Object | Role |
|--------|------|
| `panel` | Raw merge output (unchanged) |
| `panel_prep` | Filtered + winsorized + share-filled table |
| `TARGET_COL` | `virus_detected` |
| `FEATURE_COLS` | Numeric feature list for upcoming models |
| `filter_report` | Drop counts and winsor caps |

Next (still Planned on the roadmap): feature distributions, target class-balance analysis, feature importance, spatial/temporal plots, then multi-model training.

In [9]:
X = panel_prep[FEATURE_COLS].copy()
y = panel_prep[TARGET_COL].astype(int).copy()
meta = panel_prep[IDENTITY_COLS + ["vector_present", "virus_detection_count"]].copy()

handoff = {
    "X_shape": X.shape,
    "y_shape": y.shape,
    "y_positive": int(y.sum()),
    "y_positive_rate": float(y.mean()),
    "feature_cols": FEATURE_COLS,
    "target_col": TARGET_COL,
    "filter_report": filter_report,
}
display(X.describe().T)
handoff

,count,mean,std,min,25%,50%,75%,max
vector_detection_count,9026.0,6.772934,10.522180,1.0,1.0,3.000000,7.0,73.50
individual_count_sum,9026.0,380.187375,1393.679130,0.0,2.0,15.000000,123.0,10683.75
individual_count_mean,9026.0,53.719999,166.182674,0.0,1.0,4.449495,27.0,1200.00
virus_source_n_sequence,9026.0,0.006537,0.178207,0.0,0.0,0.000000,0.0,10.00
virus_source_n_outbreak,9026.0,0.007645,0.245169,0.0,0.0,0.000000,0.0,14.00
share_virus_exact_coords,9026.0,0.003619,0.059849,0.0,0.0,0.000000,0.0,1.00
share_virus_subnational_centroid,9026.0,0.000259,0.015292,0.0,0.0,0.000000,0.0,1.00


{'X_shape': (9026, 7),
 'y_shape': (9026,),
 'y_positive': 35,
 'y_positive_rate': 0.0038776866829160206,
 'feature_cols': ['vector_detection_count',
  'individual_count_sum',
  'individual_count_mean',
  'virus_source_n_sequence',
  'virus_source_n_outbreak',
  'share_virus_exact_coords',
  'share_virus_subnational_centroid'],
 'target_col': 'virus_detected',
 'filter_report': {'n_input': 12995,
  'n_pos_input': 35,
  'dropped_identity': 0,
  'dropped_bad_month': 0,
  'dropped_no_vector': 0,
  'dropped_year_window': 3969,
  'year_min': 2010,
  'year_max': None,
  'winsor_caps': {'vector_detection_count': 73.5,
   'individual_count_sum': 10683.75,
   'individual_count_mean': 1200.0},
  'winsor_q': 0.99,
  'n_output': 9026,
  'n_pos_output': 35,
  'n_dropped_total': 3969,
  'pos_retained': True}}